# Report graphs and figures

Regenerates the report figures that are **not** produced by the per-task `FINAL Step 2.x` notebooks, so every Chapter-5 figure is reproducible from a notebook:

1. **Project-wide master comparison charts** (`all_methods_{tissue,mass}_auroc.png`) — *new here*, plotted from the consolidated `report_figures/all_methods_master.csv` (whose `source` column records which FINAL notebook each row comes from).
2. **WS-map-vs-averaged-scalar illustration** (`ws_map_vs_average_example.png`) — *new here*, from the cached Step 2.4 region-aware scattering tensors.
3. **Preprocessing-sensitivity figures** — *moved here* from `scripts/make_preproc_figures.py`.
4. **Wavelet scalogram** (method illustration) — *new here*, a frequency × position scalogram of a mass profile; report Fig. 2.2.

Everything reads cached FINAL-notebook outputs (CSVs / `.npz` tensors); nothing recomputes the wavelet scattering or retrains any model. Figures are written to `data/outputs/report_figures/` and `data/outputs/recovered_region_aware_ws_cnn/`. Uses the `Agg` backend, so it runs headless and simply saves the PNGs.

## 0. Setup — imports, project root, shared style

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def find_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "data" / "outputs").is_dir():
            return c
    raise RuntimeError("project root (with data/outputs) not found")


ROOT = find_root()
O = ROOT / "data" / "outputs"
FIG = O / "report_figures"; FIG.mkdir(parents=True, exist_ok=True)
INV = O / "original_image_patch_investigation"
RA = O / "recovered_region_aware_ws_cnn"
J, L = 3, 5
REGIONS = ["full", "inner", "boundary", "context"]

BLUE, GREEN, ORANGE, PURPLE, CRIMSON, GREY, BROWN, BLACK = (
    "#4C72B0", "#55A868", "#DD8452", "#8172B3", "#C44E52", "#8C8C8C", "#937860", "#333333")
plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "savefig.dpi": 150})


def _labels(ax, bars, vals, fmt="{:.3f}", pad=0.004, fs=8):
    for b, v in zip(bars, vals):
        ax.text(b.get_width() + pad, b.get_y() + b.get_height() / 2,
                fmt.format(v), va="center", ha="left", fontsize=fs)

## 1. Project-wide master comparison charts

Two horizontal AU-ROC bar charts (tissue, mass) ranking every method, coloured by family, with across-seed error bars. Source: `report_figures/all_methods_master.csv`.

In [2]:
# ==================== 1. master comparison charts (NEW) ==================== #
def fig_master_comparison():
    df = pd.read_csv(FIG / "all_methods_master.csv")
    fam_color = {"Paper": ORANGE, "Replication": BLUE, "Extension screen": GREEN,
                 "Region-aware / spatial-map": PURPLE, "Region ablation": BROWN,
                 "Appendix / negative control": GREY, "Baseline": BLACK}
    for task in ["tissue", "mass"]:
        sub = df[df.task == task].sort_values("auroc").reset_index(drop=True)
        y = np.arange(len(sub))
        colors = [fam_color.get(f, GREY) for f in sub.family]
        fig, ax = plt.subplots(figsize=(10, max(5.0, 0.42 * len(sub))))
        bars = ax.barh(y, sub.auroc, xerr=sub.auroc_sd.fillna(0.0), capsize=2,
                       color=colors, height=0.72, error_kw=dict(lw=1, alpha=0.6))
        _labels(ax, bars, sub.auroc.values, fs=8)
        ax.set_yticks(y); ax.set_yticklabels(sub.method, fontsize=8)
        ax.set_xlim(0.45, 1.06)
        ax.set_xlabel("AU-ROC (project rows: five-seed mean $\\pm$ std; paper rows: single split)")
        ax.set_title(f"{task.capitalize()} task — all methods ranked by AU-ROC")
        ax.axvline(0.5, color="grey", lw=0.8, ls=":")
        fams = [f for f in fam_color if f in set(sub.family)]
        handles = [plt.Rectangle((0, 0), 1, 1, color=fam_color[f]) for f in fams]
        ax.legend(handles, fams, loc="lower right", fontsize=8, frameon=True)
        ax.grid(axis="x", alpha=0.3)
        fig.tight_layout()
        out = FIG / f"all_methods_{task}_auroc.png"
        fig.savefig(out, bbox_inches="tight"); plt.close(fig); print("saved", out.name)

fig_master_comparison()

saved all_methods_tissue_auroc.png


saved all_methods_mass_auroc.png


## 2. WS maps vs. averaged scalars

Illustrates why paper-style averaging discards spatial structure: the whole-lesion $J{=}3$ scattering gives 91 maps of $28\times28$ (three shown), which averaging collapses to one scalar per channel.

In [3]:
# ==================== 2. WS map vs averaged scalar (NEW) =================== #
def fig_ws_map_vs_average(fid="22614431"):
    manifest = pd.read_csv(RA / "region_aware_mass_manifest.csv")
    X = np.load(RA / f"region_aware_ws_tensors_J{J}_L{L}_full_inner_boundary_context.npz",
                allow_pickle=True)["X"]
    cpr = X.shape[1] // len(REGIONS)              # 91 channels per region
    idx = int(np.where(manifest["file_id"].astype(str).to_numpy() == fid)[0][0])
    full_maps = X[idx, 0:cpr]                     # (91, 28, 28): whole-lesion J=3 maps
    avg = full_maps.mean(axis=(1, 2))             # (91,): paper-style averaged scalars
    row = manifest.iloc[idx]
    img = np.load(row.preproc_path).astype(np.float32)
    patch = cv2.resize(img[int(row.y0):int(row.y1), int(row.x0):int(row.x1)].astype(np.float32),
                       (224, 224), interpolation=cv2.INTER_AREA)
    ex = [3, 9, 1 + J * L + 30]                   # two order-1 maps + one order-2 map

    fig = plt.figure(figsize=(14, 4.3))
    gs = fig.add_gridspec(1, 6, width_ratios=[1.35, 1, 1, 1, 0.12, 0.9], wspace=0.35)
    ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(patch, cmap="gray")
    ax0.set_title("mass patch"); ax0.axis("off")
    for k, ei in enumerate(ex):
        axk = fig.add_subplot(gs[0, 1 + k])
        axk.imshow(full_maps[ei], cmap="magma", aspect="auto")
        axk.set_title(f"WS map (J=3)\n$28\\times28$, ch. {ei}"); axk.axis("off")
    axr = fig.add_subplot(gs[0, 5])
    logavg = np.log1p(np.maximum(avg, 0.0))
    vlo, vhi = np.percentile(logavg, [4, 96])   # clip the order-0 outlier so the 91-value variation shows
    axr.imshow(logavg.reshape(-1, 1), cmap="magma", aspect="auto", vmin=vlo, vmax=vhi)
    axr.set_title("averaged WS\n(1 scalar / channel)")
    axr.set_xticks([]); axr.set_ylabel("coefficient channel (91)")
    fig.suptitle("Wavelet-scattering maps ($J{=}3$: $91\\times28\\times28$) vs.\\ the paper-style averaged scalars", y=1.03)
    out = RA / "ws_map_vs_average_example.png"
    fig.savefig(out, dpi=150, bbox_inches="tight"); plt.close(fig); print("saved", out.name)

fig_ws_map_vs_average()

saved ws_map_vs_average_example.png


## 3. Preprocessing-sensitivity figures

Moved from `scripts/make_preproc_figures.py`. Reads the Step 2.6 pixel-source and masked/no-CLAHE CSVs.

In [4]:
# ============= 3. preprocessing figures (moved from make_preproc_figures.py) ============= #
def fig_preproc_levels():
    df = pd.read_csv(INV / "ws_pixel_source_variant_summary.csv")
    order = ["current_full_preprocessed", "masked_no_clahe", "raw_percentile", "raw_minmax"]
    nice = {"current_full_preprocessed": "Current\n(percentile + CLAHE)", "masked_no_clahe": "Masked,\nno CLAHE",
            "raw_percentile": "Raw percentile", "raw_minmax": "Raw min–max"}
    y = np.arange(len(order)); h = 0.38
    fig, ax = plt.subplots(figsize=(9.2, 5.2))
    for k, (task, color) in enumerate([("tissue", BLUE), ("mass", ORANGE)]):
        sub = df[df.task == task].set_index("variant")
        vals = [float(sub.loc[s, "auroc"]) for s in order]
        sds = [float(sub.loc[s, "auroc_sd"]) for s in order]
        off = h / 2 if k == 0 else -h / 2
        bars = ax.barh(y + off, vals, height=h, xerr=sds, capsize=3, color=color,
                       label=task.capitalize(), error_kw=dict(lw=1, alpha=0.7))
        _labels(ax, bars, vals)
    ax.set_yticks(y); ax.set_yticklabels([nice[s] for s in order]); ax.invert_yaxis()
    ax.set_xlim(0.5, 1.02); ax.set_xlabel("AU-ROC (WS-only, 5-seed mean ± std)")
    ax.set_title("WS AU-ROC across preprocessing levels — tissue vs mass")
    ax.axvline(0.5, color="grey", lw=0.8, ls=":"); ax.legend(loc="lower right", frameon=True)
    ax.grid(axis="x", alpha=0.3); fig.tight_layout()
    out = FIG / "preproc_levels_ws_auroc.png"; fig.savefig(out, bbox_inches="tight"); plt.close(fig); print("saved", out.name)


def fig_paper_full_original():
    pv = pd.read_csv(FIG / "paper_vs_replication_auroc.csv")
    mt = {"tissue": pd.read_csv(INV / "master_table_tissue_original.csv"),
          "mass": pd.read_csv(INV / "master_table_mass_original.csv")}
    method_row = {"WS-only": "WS-only (averaged)", "CNN-only": "CNN-only (ResNet18→kNN)",
                  "CNN+WS fusion": "CNN+WS fusion (concat)"}
    combos = [("Tissue", "WS-only"), ("Tissue", "CNN-only"), ("Tissue", "CNN+WS fusion"),
              ("Mass", "WS-only"), ("Mass", "CNN-only"), ("Mass", "CNN+WS fusion")]

    def full(task, m):
        r = pv[(pv.task == task) & (pv.method == m) & (pv.series == "Our replication")].iloc[0]
        return float(r.auroc), float(r.sd)

    def paper(task, m):
        return float(pv[(pv.task == task) & (pv.method == m) & (pv.series == "Razali paper")].iloc[0].auroc)

    def orig(task, m):
        d = mt[task.lower()].set_index("Method")
        return float(d.loc[method_row[m], "AUROC_num"]), float(d.loc[method_row[m], "auroc_sd"])

    y = np.arange(len(combos)); h = 0.26
    fig, ax = plt.subplots(figsize=(10, 6.4))
    full_v = [full(t, m) for t, m in combos]; orig_v = [orig(t, m) for t, m in combos]; paper_v = [paper(t, m) for t, m in combos]
    b1 = ax.barh(y + h, [v for v, _ in full_v], height=h, xerr=[s for _, s in full_v], capsize=2, color=BLUE, label="Our replication (full preproc.)", error_kw=dict(lw=1, alpha=0.7))
    b2 = ax.barh(y, [v for v, _ in orig_v], height=h, xerr=[s for _, s in orig_v], capsize=2, color=GREEN, label="Original pixels (raw min–max)", error_kw=dict(lw=1, alpha=0.7))
    b3 = ax.barh(y - h, paper_v, height=h, color=ORANGE, label="Razali paper")
    _labels(ax, b1, [v for v, _ in full_v], fs=7.5); _labels(ax, b2, [v for v, _ in orig_v], fs=7.5); _labels(ax, b3, paper_v, fs=7.5)
    iws = combos.index(("Mass", "WS-only"))
    ax.axhspan(iws - 0.5, iws + 0.5, color=GREEN, alpha=0.08)
    ax.annotate("only combo where lighter\npreprocessing helps", xy=(orig_v[iws][0], iws), xytext=(0.86, iws - 0.30), fontsize=8.5, color="#2f7d32", ha="left", va="center", arrowprops=dict(arrowstyle="->", color="#2f7d32", lw=1.3), bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#2f7d32", alpha=0.9))
    ax.set_yticks(y); ax.set_yticklabels([f"{t}\n{m}" for t, m in combos]); ax.invert_yaxis()
    ax.set_xlim(0, 1.18); ax.set_xlabel("AU-ROC"); ax.set_title("Razali paper vs full preprocessing vs original pixels (raw min–max)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.07), ncol=3, frameon=True, fontsize=9); ax.grid(axis="x", alpha=0.3); fig.tight_layout()
    out = FIG / "preproc_paper_full_original_auroc.png"; fig.savefig(out, bbox_inches="tight"); plt.close(fig); print("saved", out.name)


def fig_mass_methods_no_clahe():
    story = pd.read_csv(RA / "mass_extensions_story_table.csv")
    nc_sum = pd.read_csv(INV / "region_aware_masked_no_clahe_ws_cnn_mass_summary.csv")
    nc_abl = pd.read_csv(INV / "region_ablation_masked_no_clahe_cnnhead_results.csv")
    nc_all = nc_sum[nc_sum.metric == "auroc"].iloc[0]; nc_db = nc_abl[nc_abl.combo == "drop boundary"].iloc[0]
    region_kw = ("Region-aware", "Late fusion", "Whole-patch")
    rows = []
    for _, r in story.iterrows():
        is_region = any(k in r.Method for k in region_kw)
        rows.append((r.Method, float(r.AUROC_num), float(r.auroc_sd), PURPLE if is_region else BLUE))
    rows.append(("Region-aware WS→CNN (masked, no-CLAHE) ★", float(nc_all["mean"]), float(nc_all["std"]), CRIMSON))
    rows.append(("Region ablation: drop-boundary (no-CLAHE)", float(nc_db.auroc), float(nc_db.auroc_sd), CRIMSON))
    rows.sort(key=lambda t: t[1])
    labels = [r[0] for r in rows]; vals = [r[1] for r in rows]; sds = [r[2] for r in rows]; colors = [r[3] for r in rows]
    y = np.arange(len(rows)); fig, ax = plt.subplots(figsize=(10.5, 6.6))
    bars = ax.barh(y, vals, xerr=sds, capsize=3, color=colors, height=0.72, error_kw=dict(lw=1, alpha=0.7))
    _labels(ax, bars, vals, fs=8.5); ax.set_yticks(y); ax.set_yticklabels(labels)
    ax.set_xlim(0.6, 1.02); ax.set_xlabel("AU-ROC (mass, 5-seed mean ± std)")
    ax.set_title("Mass methods — no-CLAHE lifts the region-aware model above the prior best"); ax.grid(axis="x", alpha=0.3)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (CRIMSON, PURPLE, BLUE)]
    ax.legend(handles, ["Masked / no-CLAHE (new)", "Region-aware (current preproc.)", "Other mass methods"], loc="lower right", frameon=True, fontsize=9)
    fig.tight_layout(); out = FIG / "mass_methods_no_clahe_region_aware.png"; fig.savefig(out, bbox_inches="tight"); plt.close(fig); print("saved", out.name)

fig_preproc_levels()
fig_paper_full_original()
fig_mass_methods_no_clahe()

saved preproc_levels_ws_auroc.png


saved preproc_paper_full_original_auroc.png


saved mass_methods_no_clahe_region_aware.png


## 4. Wavelet scalogram (method illustration)

A wavelet scalogram (frequency × position) of a 1-D intensity profile through an example mammographic mass, illustrating the wavelet scale-frequency decomposition — the wavelet-modulus coefficients before the scattering low-pass average. Used as report Fig. 2.2. Writes `report_figures/wavelet_scalogram.png`.

In [ ]:
# ==================== 4. Wavelet scalogram (method illustration, report Fig. 2.2) ==================== #
def fig_wavelet_scalogram(file_id=22427751, row_frac=0.5):
    RA = O / "recovered_region_aware_ws_cnn"
    r = pd.read_csv(RA / "region_aware_mass_manifest.csv").query("file_id == @file_id").iloc[0]
    im = np.load(r.preproc_path).astype(np.float32)
    patch = cv2.resize(im[int(r.y0):int(r.y1), int(r.x0):int(r.x1)].astype(np.float32), (224, 224), interpolation=cv2.INTER_AREA)

    def morlet_cwt(sig, freqs, w0=6.0):                         # analytic Morlet CWT; returns |x * psi|
        n = len(sig); sig = sig - sig.mean(); F = np.fft.fft(sig); k = np.fft.fftfreq(n)
        out = np.empty((len(freqs), n))
        for i, f in enumerate(freqs):
            s = w0 / (2 * np.pi * f); omega = 2 * np.pi * s * k
            psi = (np.pi ** -0.25) * np.exp(-0.5 * (omega - w0) ** 2); psi[k <= 0] = 0.0
            out[i] = np.abs(np.fft.ifft(F * psi * np.sqrt(2 * np.pi * s)))
        return out

    rowi = int(round(row_frac * 224)); line = patch[rowi]
    freqs = np.geomspace(0.40, 0.02, 64)
    C = morlet_cwt(line, freqs)

    fig = plt.figure(figsize=(7.6, 8.2))
    gs = fig.add_gridspec(3, 1, height_ratios=[3.0, 1.0, 2.4], hspace=0.42)
    ax0 = fig.add_subplot(gs[0]); ax0.imshow(patch, cmap="gray"); ax0.axhline(rowi, color="#22d3ee", lw=1.6)
    ax0.set_title("(a) mammogram patch with the 1-D analysis line", fontsize=10.5); ax0.set_xticks([]); ax0.set_yticks([])
    ax1 = fig.add_subplot(gs[1]); ax1.plot(line, color="#222", lw=1.1); ax1.set_xlim(0, len(line) - 1)
    ax1.set_title("(b) intensity profile along the line", fontsize=10.5); ax1.set_ylabel("intensity"); ax1.set_xticks([]); ax1.grid(alpha=0.25)
    ax2 = fig.add_subplot(gs[2])
    ax2.imshow(C, aspect="auto", cmap="magma", extent=[0, len(line), np.log10(freqs[-1]), np.log10(freqs[0])])
    ax2.set_title("(c) wavelet scalogram: frequency (y) vs position (x)", fontsize=10.5)
    ax2.set_xlabel("position along line (pixels)"); ax2.set_ylabel("frequency (cycles/pixel)")
    yt = np.log10(freqs[::12]); ax2.set_yticks(yt); ax2.set_yticklabels([f"{fr:.3f}" for fr in freqs[::12]])
    out = FIG / "wavelet_scalogram.png"; fig.savefig(out, dpi=145, bbox_inches="tight"); plt.close(fig); print("saved", out.name)

fig_wavelet_scalogram()
